<a href="https://colab.research.google.com/github/sgruyzcki/DiploDatos2026/blob/main/An%C3%A1lisis%20Exploratorio%20y%20Curaci%C3%B3n%20de%20Datos/parte%202/EyCD_2026_Entregable_Parte_2_Grupo_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Diplomatura en Ciencia de Datos, Aprendizaje Automático y sus Aplicaciones**

## **Edición 2026**
Grupo 3:

- BAGGINI, Mariano

- DA SILVA, Rocio

- GRUZYCKI, Sergio

- NEDEL, Agustina

----

# Trabajo práctico entregable - parte 2

En esta notebook, vamos a cargar el conjunto de datos de [la compentencia Kaggle](https://www.kaggle.com/dansbecker/melbourne-housing-snapshot) sobre estimación de precios de ventas de propiedades en Melbourne, Australia.

Utilizaremos el conjunto de datos reducido producido por [DanB](https://www.kaggle.com/dansbecker). Hemos subido una copia a un servidor de la Universidad Nacional de Córdoba para facilitar su acceso remoto.

In [100]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.graph_objects as go
sns.set_context('talk')

from sqlalchemy import create_engine, text

## Ejercicio 1 SQL:

1. Crear una base de datos en SQLite utilizando la libreria [SQLalchemy](https://stackoverflow.com/questions/2268050/execute-sql-from-file-in-sqlalchemy).
https://docs.sqlalchemy.org/en/14/core/engines.html#sqlite

2. Ingestar los datos provistos en 'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv' en una tabla y el dataset generado en clase con datos de airbnb y sus precios por codigo postal en otra.

3. Validar tipos de columnas antes de guardar. Usá `df.dtypes` para ver los tipos actuales. Prestá especial atención a columnas como `Date` y `Price`: por ejemplo, `Date` puede estar como string en vez de datetime, y `Price` puede venir como string o float. El método `to_sql()` infiere tipos automáticamente, pero puede fallar si los tipos no son los esperados.

4. Implementar consultas en SQL que respondan con la siguiente información:

    - cantidad de registros totales por `Regionname`.
    - cantidad de registros totales por `Suburb` y `Regionname`.
    - Consulta con filtro: ¿Cuántas propiedades hay por `Regionname` con más de 2 habitaciones?
    - Agregación condicional: ¿Cuál es el precio promedio de propiedades según tipo (`Type`) y `Regionname`?
    - Orden y límites: Mostrá el top 5 barrios con propiedades más caras en promedio.

5. Combinar los datasets de ambas tablas ingestadas utilizando el comando JOIN de SQL para obtener un resultado similar a lo realizado con Pandas en clase.

6. Agregar una celda de validación posterior al JOIN con assertions o validación de esquema. Como mínimo, verificá que el número de filas no cambió, que no aparecieron nulos inesperados y que los rangos de variables agregadas sean razonables. Esta validación implementa dimensiones básicas de calidad de datos como validez, completitud e integridad.



## Resolución Ejercicio 1

### Tratamiento del Dataset de Melbourne

In [101]:
# Dataset de propiedades de Melbourne
melb_data = pd.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/melb_data.csv')
melb_data[:3]

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Date,Distance,Postcode,...,Bathroom,Car,Landsize,BuildingArea,YearBuilt,CouncilArea,Lattitude,Longtitude,Regionname,Propertycount
0,Abbotsford,85 Turner St,2,h,1480000.0,S,Biggin,3/12/2016,2.5,3067.0,...,1.0,1.0,202.0,NaN,NaN,Yarra,-37.7996,144.9984,Northern Metropolitan,4019.0
1,Abbotsford,25 Bloomburg St,2,h,1035000.0,S,Biggin,4/02/2016,2.5,3067.0,...,1.0,0.0,156.0,79.0,1900.0,Yarra,-37.8079,144.9934,Northern Metropolitan,4019.0
2,Abbotsford,5 Charles St,3,h,1465000.0,SP,Biggin,4/03/2017,2.5,3067.0,...,2.0,0.0,134.0,150.0,1900.0,Yarra,-37.8093,144.9944,Northern Metropolitan,4019.0


In [102]:
# Estudio de propiedades del dataset
melb_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13580 entries, 0 to 13579
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         13580 non-null  object 
 1   Address        13580 non-null  object 
 2   Rooms          13580 non-null  int64  
 3   Type           13580 non-null  object 
 4   Price          13580 non-null  float64
 5   Method         13580 non-null  object 
 6   SellerG        13580 non-null  object 
 7   Date           13580 non-null  object 
 8   Distance       13580 non-null  float64
 9   Postcode       13580 non-null  float64
 10  Bedroom2       13580 non-null  float64
 11  Bathroom       13580 non-null  float64
 12  Car            13518 non-null  float64
 13  Landsize       13580 non-null  float64
 14  BuildingArea   7130 non-null   float64
 15  YearBuilt      8205 non-null   float64
 16  CouncilArea    12211 non-null  object 
 17  Lattitude      13580 non-null  float64
 18  Longti

#### Limpieza

Se siguió en gran medida el mismo proceso de limpieza realizado en la parte 1 del entregable para este dataset. La columna `Postcode` se guarda como variable entera para compatibilizar tipo con `Zipcode` del otro dataset, y se utilizó la columna `Date` para generar las columnas `SaleMonth` y `SaleYear`, en lugar de utilizar `Semester`.

In [103]:
# Copia de trabajo: no incluye BuildingArea ni YearBuilt
melb_df = melb_data.drop(['BuildingArea', 'YearBuilt'], axis=1).copy()

In [104]:
# Estudio de valores faltantes y ceros por columna
faltantes = melb_df.isna().sum()
ceros = (melb_df == 0).sum()
resumen = pd.DataFrame({'Faltantes': faltantes, 'Ceros': ceros})
resumen[resumen.any(axis=1)]

,Faltantes,Ceros
Distance,0,6
Bedroom2,0,16
Bathroom,0,34
Car,62,1026
Landsize,0,1939
CouncilArea,1369,0


Se observan 1026 registros de propiedades sin cochera, es decir con `Car = 0`. Los 62 casos en los que `Car = Null` se descartan sin pérdida de generalidad.

In [105]:
# Eliminamos columnas faltantes en la variable Car
melb_df = melb_df.dropna(subset=['Car'])

In [106]:
# Eliminamos columnas faltantes en la variable Car
melb_df = melb_df.dropna(subset=['Car'])

In [107]:
missing_values_count = melb_df.isna().sum()
missing_values_count[missing_values_count > 0]

,0
CouncilArea,1307


Como `CouncilArea` es una variable categórica, se decidió conservar los casos faltantes creando una nueva categoría `Desconocido` para no perder observaciones.

In [108]:
melb_df['CouncilArea'] = melb_df['CouncilArea'].fillna('Desconocido')
# Conteo de valores faltantes
melb_df.isna().sum()

,0
Suburb,0
Address,0
Rooms,0
Type,0
Price,0
Method,0
SellerG,0
Date,0
Distance,0
Postcode,0


In [109]:
# Estudio de los casos en los que Distance = 0
melb_df[melb_df['Distance'] == 0][['Suburb', 'Distance', 'Regionname']].head(10)

,Suburb,Distance,Regionname
9620,Melbourne,0.0,Northern Metropolitan
10393,Melbourne,0.0,Northern Metropolitan
10739,Melbourne,0.0,Northern Metropolitan
11428,Melbourne,0.0,Northern Metropolitan
12073,Melbourne,0.0,Northern Metropolitan
12074,Melbourne,0.0,Northern Metropolitan


Se analizaron los registros con `Distance = 0` y se observó que todos corresponden al suburbio Melbourne. Estos casos pueden interpretarse como propiedades ubicadas en el centro principal de la ciudad. Por lo tanto, se considera un valor válido y no un dato faltante codificado.

In [110]:
# Estudio de la relación entre Bedrooms2 y Rooms
pd.crosstab(melb_df['Bedroom2'], melb_df['Rooms'])

Rooms,1,2,3,4,5,6,7,8,10
Bedroom2,,,,,,,,,
0.0,0,5,8,3,0,0,0,0,0
1.0,660,21,5,2,0,0,0,0,0
2.0,16,3513,162,19,1,0,0,0,0
3.0,2,74,5597,175,18,1,0,0,0
4.0,0,8,73,2469,42,4,0,1,0
5.0,0,1,5,15,531,2,2,0,0
6.0,0,0,0,0,2,59,0,2,0
7.0,0,0,0,0,1,1,8,0,0
8.0,0,0,0,0,1,0,0,4,0


Se observan muchos casos en los que `Bedroom2 > Rooms`, lo cual carece de sentido. No es posible tener una cantidad de dormitorios superior al número de ambientes que tiene la propiedad. Como `Bedroom2` se obtuvo de fuentes externas y la información que aporta distorciona la interpretación del dataset, se opta por eliminarla y conservar únicamente la columna `Rooms`.

In [111]:
melb_df = melb_df.drop('Bedroom2', axis=1)

In [112]:
# Estudio de la relación entre Bathroom y Rooms
pd.crosstab(melb_df['Bathroom'], melb_df['Rooms'])

Rooms,1,2,3,4,5,6,7,8,10
Bathroom,,,,,,,,,
0.0,1,19,11,3,0,0,0,0,0
1.0,671,3093,3233,451,17,2,0,0,0
2.0,6,502,2447,1718,260,20,3,2,0
3.0,0,8,153,468,244,35,5,2,1
4.0,0,0,5,41,47,9,2,2,0
5.0,0,0,1,2,25,0,0,0,0
6.0,0,0,2,0,2,1,0,0,0
7.0,0,0,0,0,1,0,0,1,0
8.0,0,0,0,1,0,0,0,1,0


Se observan algunos puntos de interés en esta tabla:

- Casos `Bathroom > Rooms`: Extraño que una propiedad tenga más baños que ambientes, sería correcto eliminar.

- `Casos Bathroom = 0`: Puede ser un indicador de pobreza, aunque esto debe verificarse con los demás indicadores (pocos ambientes, construcción pequeña, etc). En casas grandes, se interpreta como error de tipeo.

Tratamiento: Se codifican como `NA` los casos `Bathroom = 0` para posteriormente eliminarlos. Del mismo modo, se eliminan los casos `Bathroom > Rooms`.

No se analiza el caso de ausencia de baños como posible indicador de pobreza.

In [113]:
melb_df.loc[melb_df['Bathroom'] < 1, 'Bathroom'] = pd.NA
melb_df.loc[melb_df['Bathroom'] > melb_df['Rooms'], 'Bathroom'] = pd.NA
melb_df = melb_df.dropna(subset=['Bathroom'])
melb_df.shape[0]

13456

Los casos `Landsize = 0` se consideran como dato faltante codificado. Tratamiento: se eliminan. Luego se eliminan outliers extremos utilizando percentiles.

También se eliminan las filas que no tienen precio, es decir `Price = 0` o `Price = Null`.

In [114]:
# Descartar casos donde Landsize = 0
melb_df = melb_df[melb_df['Landsize'] != 0]

# Eliminar outliers extremos
q99_landsize = melb_df['Landsize'].quantile(0.99)
melb_df = melb_df[(melb_df['Landsize'] < q99_landsize) | (melb_df['Landsize'].isnull())]

Análisis de variables categóricas: Se verifican los tipos de datos asignados y se hacen las correcciones necesarias.

In [115]:
# Verificacion de types en las variables
melb_df.dtypes

,0
Suburb,object
Address,object
Rooms,int64
Type,object
Price,float64
Method,object
SellerG,object
Date,object
Distance,float64
Postcode,float64


Corrección de tipos de variables: `Propertycount`, `Bathroom` y `Car` deben ser variables numéricas enteras en lugar de reales. `Postcode` debe ser una variable compatible con la variable `zipcode` del dataset de AirBnB, por lo que se convierte también en numérica entera. `Date` se convierte a fecha.

In [116]:
# Postcode como integer
melb_df['Postcode'] = melb_df['Postcode'].astype(int)

# Propertycount como integer
melb_df['Propertycount'] = melb_df['Propertycount'].astype(int)

# Car como integer
melb_df['Car'] = melb_df['Car'].astype(int)

# Bathroom como integer
melb_df['Bathroom'] = melb_df['Bathroom'].astype(int)

# Date como fecha
melb_df['Date'] = pd.to_datetime(melb_df['Date'], dayfirst=True)

melb_df.dtypes

,0
Suburb,object
Address,object
Rooms,int64
Type,object
Price,float64
Method,object
SellerG,object
Date,datetime64[ns]
Distance,float64
Postcode,int64


In [117]:
# Se discrimina el mes y el año de la venta en columnas independientes
melb_df['SaleYear']  = melb_df['Date'].dt.year
melb_df['SaleMonth'] = melb_df['Date'].dt.month

# Descartar la columna original
melb_df = melb_df.drop(columns=['Date'], axis=1)

melb_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 11423 entries, 0 to 13579
Data columns (total 19 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Suburb         11423 non-null  object 
 1   Address        11423 non-null  object 
 2   Rooms          11423 non-null  int64  
 3   Type           11423 non-null  object 
 4   Price          11423 non-null  float64
 5   Method         11423 non-null  object 
 6   SellerG        11423 non-null  object 
 7   Distance       11423 non-null  float64
 8   Postcode       11423 non-null  int64  
 9   Bathroom       11423 non-null  int64  
 10  Car            11423 non-null  int64  
 11  Landsize       11423 non-null  float64
 12  CouncilArea    11423 non-null  object 
 13  Lattitude      11423 non-null  float64
 14  Longtitude     11423 non-null  float64
 15  Regionname     11423 non-null  object 
 16  Propertycount  11423 non-null  int64  
 17  SaleYear       11423 non-null  int32  
 18  SaleMonth  

In [118]:
# Cantidad de ventas realizadas agrupadas por mes y discriminadas por año
melb_df.groupby(['SaleYear', 'SaleMonth']).size()

SaleYear  SaleMonth
2016      1               2
          2              22
          4             254
          5             706
          6             569
          7             350
          8             562
          9             732
          10            441
          11            889
          12            494
2017      2             347
          3             547
          4             509
          5            1004
          6            1000
          7            1355
          8             773
          9             867
dtype: int64

#### Resultado: Limpieza del dataset de Melbourne.

El dataset `melb_df` no tiene valores nulos; se descartó la columna `Bedroom2` y las entradas en las que:

- `Bathroom = 0`
- `Bathroom > Rooms`
- `Landsize = 0`

También se hizo un filtro de outliers al 99% en `Landsize`.

Se codificaron valores nulos de `CouncilArea` como `Desconocido`.

Se descartó la columna `Date`, codificando su información en las columnas `SaleYear` y `SaleMonth`.

Además, todos los tipos de datos fueron debidamente corregidos.

### Tratamiento del Dataset de AirBnB

In [119]:
# Dataset de propiedades de AirBnB (completo)
airbnb_data = pd.read_csv(
    'https://cs.famaf.unc.edu.ar/~mteruel/datasets/diplodatos/cleansed_listings_dec18.csv')
airbnb_data[:3]

/tmp/ipykernel_4375/1353126128.py:2: DtypeWarning:

Columns (35,77) have mixed types. Specify dtype option on import or set low_memory=False.



,id,listing_url,scrape_id,last_scraped,name,summary,space,description,neighborhood_overview,notes,...,review_scores_location,review_scores_value,requires_license,license,instant_bookable,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,reviews_per_month
0,9835,https://www.airbnb.com/rooms/9835,2.018120e+13,12/7/2018,Beautiful Room & House,NaN,"House: Clean, New, Modern, Quite, Safe. 10Km f...","House: Clean, New, Modern, Quite, Safe. 10Km f...",Very safe! Family oriented. Older age group.,NaN,...,9.0,9.0,f,NaN,f,strict_14_with_grace_period,f,f,1,0.04
1,10803,https://www.airbnb.com/rooms/10803,2.018120e+13,12/7/2018,Room in Cool Deco Apartment in Brunswick,A large air conditioned room with queen spring...,The apartment is Deco/Edwardian in style and h...,A large air conditioned room with queen spring...,This hip area is a crossroads between two grea...,NaN,...,9.0,9.0,f,NaN,t,moderate,t,t,1,1.50
2,12936,https://www.airbnb.com/rooms/12936,2.018120e+13,12/7/2018,St Kilda 1BR APT+BEACHSIDE+VIEWS+PARKING+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,FREE WiFi FREE in-building remote controlled g...,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,A stay at our apartment means you can enjoy so...,First floor apartment with both lift and stair...,...,9.0,9.0,f,NaN,f,strict_14_with_grace_period,f,f,17,0.15


El dataset completo tiene 84 columnas, de las cuales nos interesa sólo un número reducido. Guardamos una copia de trabajo con el filtro aplicado.

In [120]:
# Columnas de interes
interesting_cols = [
  'description', 'neighborhood_overview',
  'street', 'neighborhood', 'city', 'suburb', 'state', 'zipcode',
  'price', 'weekly_price', 'monthly_price',
  'latitude', 'longitude',
]

In [121]:
# Guardo una copia del dataset original solo con las columnas de interes
airbnb_df = airbnb_data[interesting_cols].copy()
airbnb_df[:3]

,description,neighborhood_overview,street,neighborhood,city,suburb,state,zipcode,price,weekly_price,monthly_price,latitude,longitude
0,"House: Clean, New, Modern, Quite, Safe. 10Km f...",Very safe! Family oriented. Older age group.,"Bulleen, VIC, Australia",Balwyn North,Manningham,Bulleen,VIC,3105,60,NaN,NaN,-37.772684,145.092133
1,A large air conditioned room with queen spring...,This hip area is a crossroads between two grea...,"Brunswick East, VIC, Australia",Brunswick,Moreland,Brunswick East,VIC,3057,35,200.0,803.0,-37.766505,144.980736
2,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,A stay at our apartment means you can enjoy so...,"St Kilda, VIC, Australia",St Kilda,Port Phillip,St Kilda,VIC,3182,159,1253.0,4452.0,-37.859755,144.977369


In [122]:
# Estudio de propiedades del dataset
airbnb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22895 entries, 0 to 22894
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   description            22563 non-null  object 
 1   neighborhood_overview  14424 non-null  object 
 2   street                 22895 non-null  object 
 3   neighborhood           17082 non-null  object 
 4   city                   22895 non-null  object 
 5   suburb                 22872 non-null  object 
 6   state                  22834 non-null  object 
 7   zipcode                22753 non-null  object 
 8   price                  22895 non-null  int64  
 9   weekly_price           2524 non-null   float64
 10  monthly_price          1891 non-null   float64
 11  latitude               22895 non-null  float64
 12  longitude              22895 non-null  float64
dtypes: float64(4), int64(1), object(8)
memory usage: 2.3+ MB


#### Limpieza

Se procuró alterar el dataset lo mínimo indispensable para poder concatenarlo con el de Melbourne sin que se produzcan incompatibilidades: Se corrigieron tipos de datos y se completaron con ceros los valores faltantes de `zipcode`.

In [123]:
# Estudio de valores faltantes y ceros por columna
faltantes_airbnb = airbnb_df.isna().sum()
ceros_airbnb = (airbnb_df == 0).sum()
resumen = pd.DataFrame({'Faltantes': faltantes_airbnb, 'Ceros': ceros_airbnb})
resumen[resumen.any(axis=1)]

,Faltantes,Ceros
description,332,0
neighborhood_overview,8471,0
neighborhood,5813,0
suburb,23,0
state,61,0
zipcode,142,0
price,0,21
weekly_price,20371,0
monthly_price,21004,0


Se observan 146 valores faltantes en la columna `zipcode`. Para compatibilizar datasets, se completan con ceros y luego se convierte la variable en entera.

In [124]:
# Conversion de tipos para compatibilizar datasets
airbnb_df['zipcode'] = pd.to_numeric(airbnb_df.zipcode, errors='coerce')
airbnb_df['zipcode'] = airbnb_df.zipcode.fillna(0).astype('int')
airbnb_df.dtypes

,0
description,object
neighborhood_overview,object
street,object
neighborhood,object
city,object
suburb,object
state,object
zipcode,int64
price,int64
weekly_price,float64


In [125]:
airbnb_df[airbnb_df['zipcode'] == 0].shape

(146, 13)

### Unificación de datasets

#### Intersección de datos

Antes de unir los conjuntos de datos, nos aseguramos de tener suficientes registros en común para agregar información relevante.

In [126]:
# Exploramos valores en comun
intersection = np.intersect1d(
    airbnb_df['zipcode'].values, melb_df['Postcode'].values, assume_unique=False)

print("Airbnb unique zipcodes:", len(airbnb_df['zipcode'].unique()))
print("Sales unique zipcodes:", len(melb_df['Postcode'].unique()))
print("Common zipcodes:", len(intersection))
print('')

print('Registros en Melbourne con zipcode en Airbnb:',
      melb_df['Postcode'].isin(intersection).sum() / len(melb_df))
print('Registros en Airbnb con zipcode en Melbourne:',
      airbnb_df['zipcode'].isin(intersection).sum() / len(airbnb_df))

Airbnb unique zipcodes: 248
Sales unique zipcodes: 194
Common zipcodes: 188

Registros en Melbourne con zipcode en Airbnb: 0.9985993171671189
Registros en Airbnb con zipcode en Melbourne: 0.9029919196331077


#### Exploración visual

Para asegurarnos de que las áreas representadas por ambos datasets son consitentes, representamos las coordenadas en un mapa usando Plotly.

In [127]:
# Exploracion visual: Distirbucion georgafica de ambos datasets

# Muestreo de Melbourne y conversión de SaleYear a string (categórico)
data_melb = melb_df.sample(300, random_state=42).copy()
data_melb['SaleYear'] = data_melb['SaleYear'].astype(str)

# Limpieza y filtrado de Airbnb
airbnb_df['state_clean'] = airbnb_df['state'].str.strip().str.upper()
airbnb_df['state_clean'] = airbnb_df['state_clean'].replace({
    'VI': 'VIC',
    'VICTORIA': 'VIC',
    '維多利亞 VIC': 'VIC'
})
# Filtramos para asegurarnos de que solo graficamos la zona de interés (VIC)
data_airbnb = airbnb_df[airbnb_df['state_clean'] == 'VIC'].sample(300, random_state=42)


# CONSTRUCCIÓN DEL MAPA UNIFICADO

fig = go.Figure()

# Capa 1: Propiedades en Venta (Melbourne)
# Usamos un bucle para separar por año de venta y evitar el gradiente continuo
for year in sorted(data_melb['SaleYear'].unique()):
    df_year = data_melb[data_melb['SaleYear'] == year]

    fig.add_trace(go.Scattermapbox(
        lat=df_year['Lattitude'],
        lon=df_year['Longtitude'],
        mode='markers',
        marker=dict(
            size=8,
            opacity=0.7
        ),
        name=f'Venta Melb - {year}',
        text=df_year.apply(lambda r: f"Precio: ${r['Price']:,.0f}<br>Tipo: {r['Type']}<br>Barrio: {r['Suburb']}", axis=1),
        hoverinfo='text'
    ))

# Capa 2: Listings de Airbnb
fig.add_trace(go.Scattermapbox(
    lat=data_airbnb['latitude'],
    lon=data_airbnb['longitude'],
    mode='markers',
    marker=dict(
        size=6,
        color='gold', # Color fijo contrastante para diferenciarlo de las ventas
        opacity=0.6
    ),
    name='Airbnb Listings',
    text=data_airbnb.apply(lambda r: f"Precio/Noche: ${r['price']}<br>Barrio: {r['neighborhood']}", axis=1),
    hoverinfo='text'
))


# DISEÑO Y CONFIGURACIÓN DEL LAYOUT

# Calculamos el centro dinámico del mapa basado en los datos de Melbourne
center_lat = data_melb['Lattitude'].median()
center_lon = data_melb['Longtitude'].median()

fig.update_layout(
    title={
        'text': "<b>Análisis Inmobiliario en Melbourne: Ventas vs. Airbnb</b>",
        'y':0.95,
        'x':0.5,
        'xanchor': 'center',
        'yanchor': 'top'
    },
    mapbox=dict(
        style='open-street-map',
        center=dict(lat=center_lat, lon=center_lon),
        zoom=9.5
    ),
    margin={"r": 10, "t": 60, "l": 10, "b": 10},
    height=650,
    legend=dict(
        title_text="<b>Capas de Datos</b>",
        yanchor="top",
        y=0.99,
        xanchor="left",
        x=0.01,
        bgcolor="rgba(255, 255, 255, 0.8)" # Fondo semi-transparente para legibilidad
    )
)

fig.show()

#### Agrupación y unificación

Se renombran y agrupan las columnas de `airbnb_df` por zipcode usando diccionarios, cuyas *keys* son las columnas originales por agregar y los *values* son las operaciones.

In [128]:
# Agrupacion por zipcode
relevant_cols = ['price', 'weekly_price', 'monthly_price', 'zipcode']
airbnb_price_by_zipcode = airbnb_df[relevant_cols].groupby('zipcode')\
  .agg({'price': ['mean', 'count'], 'weekly_price': 'mean',
        'monthly_price': 'mean'})\
  .reset_index()

# Flatten the two level columns
airbnb_price_by_zipcode.columns = [
  ' '.join(col).strip()
  for col in airbnb_price_by_zipcode.columns.values]

# Renombrar columnas
airbnb_price_by_zipcode = airbnb_price_by_zipcode.rename(
    columns={'price mean': 'airbnb_price_mean',
             'price count': 'airbnb_record_count',
             'weekly_price mean': 'airbnb_weekly_price_mean',
             'monthly_price mean': 'airbnb_monthly_price_mean'}
)

print(f"Dimensiones finales: {airbnb_price_by_zipcode.shape}")

Dimensiones finales: (248, 5)


In [129]:
# Verificacion de columnas contenidas
airbnb_price_by_zipcode.columns

Index(['zipcode', 'airbnb_price_mean', 'airbnb_record_count',
       'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'],
      dtype='object')

In [130]:
airbnb_price_by_zipcode[:3]

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,0,159.630137,146,906.083333,3179.666667
1,2010,40.000000,1,NaN,NaN
2,2134,50.000000,1,NaN,NaN


In [131]:
# Guardado del dataset en formato .csv
airbnb_price_by_zipcode.to_csv("airbnb_price_by_zipcode.csv", index=None)

In [132]:
# Unificacion del dataset de Melbourne con el de airbnb agrupado por zipcode
merged_sales_df = melb_df.merge(
    airbnb_price_by_zipcode, how='left',
    left_on='Postcode', right_on='zipcode'
)
merged_sales_df.sample(5)

,Suburb,Address,Rooms,Type,Price,Method,SellerG,Distance,Postcode,Bathroom,...,Longtitude,Regionname,Propertycount,SaleYear,SaleMonth,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
7121,Kings Park,169 Gillespie Rd,3,h,520000.0,PI,Bells,14.0,3021,1,...,144.77161,Western Metropolitan,2878,2017,7,3021.0,81.650000,20.0,350.000000,1300.000000
2321,Glen Iris,28 Ferndale Rd,3,h,2200000.0,S,Marshall,9.2,3146,1,...,145.07110,Southern Metropolitan,10412,2017,2,3146.0,135.833333,72.0,770.111111,2940.750000
4677,Strathmore,45 Loeman St,4,h,1280000.0,VB,Brad,9.7,3041,2,...,144.91960,Western Metropolitan,3284,2017,3,3041.0,106.125000,8.0,350.000000,1500.000000
1221,Brunswick West,497 Albion St,5,h,1350000.0,SP,Woodards,5.9,3055,2,...,144.94140,Northern Metropolitan,7082,2016,11,3055.0,87.827586,87.0,519.285714,1725.933333
1599,Chadstone,15 Aloomba St,3,h,992000.0,S,Gary,13.6,3148,1,...,145.09780,Southern Metropolitan,3582,2016,10,3148.0,79.500000,26.0,197.500000,1660.333333


In [133]:
merged_sales_df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Distance', 'Postcode', 'Bathroom', 'Car', 'Landsize', 'CouncilArea',
       'Lattitude', 'Longtitude', 'Regionname', 'Propertycount', 'SaleYear',
       'SaleMonth', 'zipcode', 'airbnb_price_mean', 'airbnb_record_count',
       'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'],
      dtype='object')

#### Validación

Se chequea que la cantidad de filas se mantuvo y que no aparecieron nulos inesperados en las columnas agregadas.

Se validan dimensiones básicas de calidad de datos:

- validez,
- completitud,
- integridad.

In [134]:
print(f"Filas antes: {len(melb_df)}")
print(f"Filas después: {len(merged_sales_df)}")
print(f"Nulls nuevos en airbnb_price_mean: {merged_sales_df['airbnb_price_mean'].isna().sum()}")

assert len(merged_sales_df) == len(melb_df), "El merge cambió el número de filas"
assert merged_sales_df["Price"].isna().sum() == 0, "Hay nulls inesperados en Price"
assert merged_sales_df["airbnb_price_mean"].dropna().between(0, 10000).all(), "Precios fuera de rango"

Filas antes: 11423
Filas después: 11423
Nulls nuevos en airbnb_price_mean: 16


In [135]:
merged_sales_df.to_csv("melb_data_extended.csv", index=None)

## Ejercicio 2 - Pandas:

Este ejercicio usa el archivo `airbnb_price_by_zipcode.csv` generado en el notebook `02.1 Combinación de datasets.ipynb`. Si no lo tenés, generarlo primero antes de comenzar esta parte.

1. Seleccionar un subconjunto de columnas que les parezcan relevantes al problema de predicción del valor de la propiedad. Justificar explicitamente las columnas seleccionadas y las que no lo fueron.
  1. Valores faltantes: ¿Qué porcentaje de filas tienen al menos un valor faltante?
  2. Mostrar la dispersión o distribución de las columnas seleccionadas.
 3. Eliminar los valores extremos que no sean relevantes para la predicción de valores de las propiedades.
 4. Mostrar visualmente los valores extremos que eliminás


2. Agregar información adicional respectiva al entorno de una propiedad a partir del [conjunto de datos de AirBnB](https://www.kaggle.com/tylerx/melbourne-airbnb-open-data?select=cleansed_listings_dec18.csv) utilizado en el práctico.
  1. Seleccionar qué variables agregar y qué combinaciones aplicar a cada una. Por ejemplo, pueden utilizar solo la columna `price`, o aplicar múltiples transformaciones como la mediana (¿por qué no la media?) o el mínimo.
  2. Utilizar la variable zipcode para unir los conjuntos de datos. Sólo incluir los zipcodes que tengan una cantidad mínima de registros (a elección) como para que la información agregada sea relevante.
  3. Mostrar un gráfico zipcode vs airbnb_price_median.
  4. Investigar al menos otras 2 variables que puedan servir para combinar los datos, y justificar si serían adecuadas o no. Pueden asumir que cuentan con la ayuda de anotadores expertos para encontrar equivalencias entre barrios o direcciones, o que cuentan con algoritmos para encontrar las n ubicaciones más cercanas a una propiedad a partir de sus coordenadas geográficas. **NO** es necesario que realicen la implementación. Si tuvieras que entrevistar a un experto inmobiliario para mapear barrios entre datasets, ¿qué 3 preguntas le harías para validar esa correspondencia?
  5. Si las coordenadas geoespaciales estuvieran disponibles, como las usarian?

Pueden leer otras columnas del conjunto de AirBnB además de las que están en `interesting_cols`, si les parecen relevantes.

¿Qué cosas no están en los datos que te gustaría tener para predecir mejor el precio de una propiedad?


In [ ]:
airbnb_price_by_zipcode.columns

Index(['zipcode', 'airbnb_price_mean', 'airbnb_record_count',
       'airbnb_weekly_price_mean', 'airbnb_monthly_price_mean'],
      dtype='object')

In [ ]:
airbnb_price_by_zipcode.head()

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
0,2010.0,40.000000,1,NaN,NaN
1,2134.0,50.000000,1,NaN,NaN
2,2582.0,104.000000,1,NaN,NaN
3,3000.0,150.504307,3367,918.738956,3407.204651
4,3001.0,132.500000,2,NaN,NaN


In [ ]:
airbnb_price_by_zipcode.describe()

,zipcode,airbnb_price_mean,airbnb_record_count,airbnb_weekly_price_mean,airbnb_monthly_price_mean
count,247.000000,247.000000,247.000000,184.000000,168.000000
mean,3508.538462,149.900412,92.101215,756.381199,2804.327037
std,1903.914390,84.872988,261.914701,393.460560,1642.559997
min,2010.000000,37.000000,1.000000,133.000000,527.000000
25%,3072.500000,93.744681,8.500000,463.721429,1645.783333
50%,3148.000000,126.012987,27.000000,667.900000,2359.000000
75%,3589.500000,187.168478,73.000000,994.500000,3450.363636
max,30122.000000,759.083333,3367.000000,2236.666667,10060.000000


### Criterios de evaluación
Se evaluará principalmente:
- claridad del código,
- justificación de las decisiones de curación,
- coherencia entre el análisis realizado y las conclusiones,
- presencia de validaciones después de operaciones críticas como merges o cargas a base.

No se espera una única solución correcta, pero sí que las decisiones estén justificadas y sean consistentes con los datos.


## Ejercicio 3:

Crear y guardar un nuevo conjunto de datos con todas las transformaciones realizadas anteriormente.

## Ejercicios opcionales:

El notebook `02.2 ETLs-DAGs.ipynb` tiene un esqueleto de referencia para guiarse.

1. Armar un script en python (archivo .py) [ETL](https://towardsdatascience.com/what-to-log-from-python-etl-pipelines-9e0cfe29950e) que corra los pasos de extraccion, transformacion y carga, armando una funcion para cada etapa del proceso y luego un main que corra todos los pasos requeridos.

2. Armar un DAG en Apache Airflow que corra el ETL. (https://airflow.apache.org/docs/apache-airflow/stable/tutorial.html)

3. Bonus: embeddings y búsqueda semántica con descripciones de AirBnB.
   - Usar `sentence-transformers` para codificar descripciones textuales de propiedades.
   - Tomar un subconjunto chico de descripciones, calcular embeddings y encontrar el par más similar con similitud coseno.
   - Reflexionar: ¿por qué este resultado no se puede lograr con `LIKE '%keyword%'` en SQL? ¿Qué pasa si dos propiedades son similares pero usan palabras distintas? ¿Qué representan los 384 números del embedding?

4. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?


5. Bonus: curación asistida por IA sobre `CouncilArea`.
   - Tomar los valores únicos de `CouncilArea` y pedirle a un modelo de lenguaje que sugiera posibles inconsistencias, duplicados por capitalización o spelling y agrupaciones razonables.
   - Proponer una estandarización preliminar y luego validar manualmente si el mapeo tiene sentido.
   - Reflexionar: ¿cuándo confiarías en este resultado sin revisarlo? ¿Qué pasa si el modelo inventa un mapeo incorrecto? ¿Cómo validarías el resultado si la columna tuviera miles de valores únicos?

Ejemplo conceptual:


In [ ]:
import anthropic
import json

client = anthropic.Anthropic()

# Valores únicos con posibles inconsistencias
council_values = melb_df['CouncilArea'].dropna().unique().tolist()

message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{
        "role": "user",
        "content": f"""Estos son los valores únicos de la columna CouncilArea en un dataset de propiedades de Melbourne:
{council_values}

Identificá: (1) duplicados con distinta capitalización o spelling,
 (2) valores que parecen errores, (3) valores que podrían agruparse.
Respondé en JSON con la estructura: {{"estandarizado": {{"valor_original": "valor_correcto"}}}}"""
    }]
)

mapping = json.loads(message.content[0].text)
melb_df['CouncilArea_clean'] = melb_df['CouncilArea'].map(
    mapping.get('estandarizado', {})
).fillna(melb_df['CouncilArea'])
